# EfficientAD and Patchcore implementation

## Setup
Before launching the training and inference scripts it's necessary to clone the project repository and to update the paths configuration inserting the right paths for input and output.

In [ ]:
!git clone https://github.com/emanuelepietrocometti/anomaly_detection_for_textile_industry.git
%cd anomaly_detection_for_textile_industry

In [ ]:
!pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130

In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Tesi/MVTec"
COLAB_DESTINATION = "/content/anomaly_detection_for_textile_industry/data/mvtec"

os.makedirs(COLAB_DESTINATION, exist_ok=True)

!cp -rp "{DATASET_PATH}"/* "{COLAB_DESTINATION}"

## Training
This script allows you to run EfficientAD and PatchCore directly in Google Colab. Before execution, you need to set up your Google Drive folder structure. Specifically, upload your dataset using the MVTec AD format, including the custom category you wish to analyze. Once uploaded, update the config.yaml file to specify the correct paths for both the input dataset and the output folders.

In [ ]:
%cd anomaly_detection_for_textile_industry

!python main.py --baseline efficientad --timestamp 29062026_084015_withDustInTrain --mode supervised

In [ ]:
import os
import shutil

RESULT_PATH = "/content/anomaly_detection_for_textile_industry/results/EfficientAd"
EXPORT_PATH = "/content/anomaly_detection_for_textile_industry/exports"
DRIVE_DESTINATION_RESULT = "/content/drive/MyDrive/Tesi/results/EfficientAD/results"
DRIVE_DESTINATION_EXPORT = "/content/drive/MyDrive/Tesi/results/EfficientAD/exports"

os.makedirs(DRIVE_DESTINATION_RESULT, exist_ok=True)
os.makedirs(DRIVE_DESTINATION_EXPORT, exist_ok=True)

if os.path.exists(RESULT_PATH):
    for item in os.listdir(RESULT_PATH):
        s = os.path.join(RESULT_PATH, item)
        d = os.path.join(DRIVE_DESTINATION_RESULT, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, symlinks=True, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print("Copy completed successfully!")

elif os.path.exists(EXPORT_PATH):
    for item in os.listdir(EXPORT_PATH):
        s = os.path.join(EXPORT_PATH, item)
        d = os.path.join(DRIVE_DESTINATION_EXPORT, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, symlinks=True, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print("Copy completed successfully!")
else:
    print(f"Error: Source path {RESULT_PATH} does not exist. Check the spelling!")

## Hyperparameter optimization

In [ ]:
%cd anomaly_detection_for_textile_industry

!python src/hyperparameter_optimization_efficientad.py

## Inference
This second script enables you to run model inference and evaluate the overall performance of the pipeline. The arguments are:
- model path: onnx model path used for the inference;
- image path: path of the image used for the inference. The program use the same image for all the batches. This is done beacase the goal of this analysis is only the inference time;
- iterations: number of iterations for each batch;
- device: allow to chose the acceleration device between 'cpu', 'cuda' and 'tensorrt'.

In [ ]:
%cd anomaly_detection_for_textile_industry

!python inference.py --model <path> --image <path> --iterations 1000 --device tensorrt